# Take-Private LBO Screen: Domino's Pizza (DPZ)

A sponsor-style LBO walkthrough using `valuationengine`.

## Thesis question

Domino's has the kind of profile that draws sponsor interest: a well-known consumer brand, predictable franchise-driven cash flows, and a long history of returning capital. The question for a PE buyer is mechanical: **at what entry multiple and capital structure does an LBO clear a 20% sponsor IRR over a five-year hold?**

We use `valuationengine`'s LBO module to answer that, including the debt schedule with mandatory amortization and a cash sweep.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from valuationengine.data.fetcher import fetch_company
from valuationengine.core.models import Assumptions
from valuationengine.core import lbo, sensitivity

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
dpz = fetch_company("DPZ")
print(f"Company: {dpz.name}")
print(f"Price: ${dpz.current_price:,.2f}")
print(f"Market cap: ${dpz.market_cap/1e9:,.1f}B")
print(f"Latest EBITDA: ${dpz.latest_ebitda/1e6:,.0f}M")
print(f"Net debt: ${dpz.net_debt/1e6:,.0f}M")
print(f"Avg operating margin: {dpz.avg_operating_margin*100:.1f}%")

## 1. Base LBO

Sponsor case: 11x entry EV/EBITDA, 60% debt at an 8% rate, 5-year hold, 11x exit. Mandatory amortization at 5% of original debt per year, 75% cash sweep on excess FCF.

In [ ]:
a = Assumptions(
    entry_ev_ebitda_multiple=11.0,
    exit_lbo_ev_ebitda_multiple=11.0,
    debt_pct_purchase=0.60,
    lbo_debt_interest_rate=0.08,
    revenue_growth=0.06,
    operating_margin=0.18,
    hold_period_years=5,
    mandatory_amortization_pct=0.05,
    cash_sweep_pct=0.75,
)
result = lbo.run(dpz, a)
print(result.summary())

## 2. Sources and uses

In [ ]:
pd.Series(result.sources_and_uses)

## 3. Debt schedule

How the cap stack delevers across the hold period.

In [ ]:
result.debt_schedule

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(result.debt_schedule["year"], result.debt_schedule["ending_balance"]/1e6, marker="o")
ax.set_xlabel("Year"); ax.set_ylabel("Ending debt balance ($M)")
ax.set_title("DPZ LBO debt paydown")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Entry-vs-exit multiple sensitivity

The single biggest driver of sponsor returns is the spread between entry and exit multiples. Hold every operating assumption constant and grid across both.

In [ ]:
entry_vals = [9.0, 10.0, 11.0, 12.0, 13.0]
exit_vals = [9.0, 10.0, 11.0, 12.0, 13.0]
irr_grid = sensitivity.run(
    dpz, a,
    x_field="entry_ev_ebitda_multiple", x_values=entry_vals,
    y_field="exit_lbo_ev_ebitda_multiple", y_values=exit_vals,
    output="irr",
    valuation_fn=lbo.run,
)
(irr_grid * 100).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(irr_grid.values * 100, aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(len(entry_vals))); ax.set_xticklabels([f"{v:.0f}x" for v in entry_vals])
ax.set_yticks(range(len(exit_vals))); ax.set_yticklabels([f"{v:.0f}x" for v in exit_vals])
ax.set_xlabel("Entry EV/EBITDA"); ax.set_ylabel("Exit EV/EBITDA")
ax.set_title("Sponsor IRR (%)")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## Takeaway

The grid makes the deal economics legible at a glance. Multiple expansion (exit > entry) drives most of the upside; entry discipline drives the rest. The debt schedule shows whether the cap stack is sustainable through the hold period. With this scaffolding, the rest of the diligence is qualitative: is the franchise system resilient through a recession, is management willing to take the deal, is the lender market open.